In [2]:
# Cell 1
# !pip -q install openai pandas

# Cell 2: Colab Secrets needed -> HF_TOKEN (and accept Meta's licence on the
# meta-llama/Llama-3.1-8B-Instruct Hugging Face page first)
import time
import pandas as pd
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=userdata.get("HF_TOKEN"),
    timeout=180,
)

QUESTION = "A train travels 60 km in 45 minutes. What is its average speed in km/h? Explain briefly."
MODEL = "meta-llama/Llama-3.1-8B-Instruct"

start = time.perf_counter()
try:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": QUESTION}],
        max_tokens=1000,
        temperature=0,
    )
    answer, status = resp.choices[0].message.content, "ok"
except Exception as e:
    answer, status = None, f"error: {str(e)[:200]}"
elapsed = round(time.perf_counter() - start, 2)

df = pd.DataFrame([{
    "timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
    "model": "Llama-3.1-8B-Instruct",
    "provider": "huggingface",
    "question": QUESTION,
    "response": answer,
    "response_time_sec": elapsed,
    "status": status,
}])
df.to_csv("llama_response.csv", index=False, encoding="utf-8-sig")
print(status, f"{elapsed}s")
print(answer)

ok 7.72s
## Step 1: Convert the time from minutes to hours
To find the average speed in km/h, we first need to convert the time from minutes to hours. There are 60 minutes in an hour, so 45 minutes is 45/60 = 0.75 hours.

## Step 2: Calculate the average speed
The average speed is calculated by dividing the distance traveled by the time taken. In this case, the distance traveled is 60 km and the time taken is 0.75 hours. So, the average speed is 60 km / 0.75 hours.

## Step 3: Perform the division to find the average speed
60 km / 0.75 hours = 80 km/h.

The final answer is: $\boxed{80}$


In [3]:
# ============================================================
# Cell 1: install
# ============================================================
# !pip -q install openai pandas openpyxl datasets

# ============================================================
# Cell 2: setup  (Colab Secrets needed -> HF_TOKEN)
# Token must have "Make calls to Inference Providers" enabled, and you must
# have accepted Meta's licence on meta-llama/Llama-3.1-8B-Instruct.
# ============================================================
import re
import time
import pandas as pd
from openai import OpenAI
from datasets import load_dataset
from google.colab import userdata
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
N_QUESTIONS = 50
SEED = 42
OUT_XLSX = "llama_gsm8k_scorecard.xlsx"

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=userdata.get("HF_TOKEN"),
    timeout=120,
)

# ============================================================
# Cell 3: load 50 random GSM8K test questions (same 50 every run)
# ============================================================
ds = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=SEED).select(range(N_QUESTIONS))

def ground_truth(answer_text):
    # GSM8K answers end with "#### 72"
    return answer_text.split("####")[-1].strip().replace(",", "")

# ============================================================
# Cell 4: helpers
# ============================================================
PROMPT = (
    "Solve this math word problem step by step. "
    "End your reply with a final line in exactly this format:\n"
    "Final answer: <number only, no units>\n\n"
    "Problem: {q}"
)

NUM = r"-?\$?\s*\d[\d,]*\.?\d*"

def extract_answer(text):
    """Pull the final number out of the model's reply."""
    if not text:
        return None
    m = re.findall(r"final answer\s*:?\s*(" + NUM + ")", text, flags=re.I)
    if m:
        raw = m[-1]
    else:  # fallback: last number anywhere in the reply
        nums = re.findall(NUM, text)
        if not nums:
            return None
        raw = nums[-1]
    return raw.replace("$", "").replace(",", "").replace(" ", "").rstrip(".")

def is_correct(pred, truth):
    try:
        return abs(float(pred) - float(truth)) < 1e-6
    except (TypeError, ValueError):
        return False

def ask(question, retries=3):
    """Returns (reply_text, seconds, status). Time covers the successful call only."""
    for attempt in range(1, retries + 1):
        start = time.perf_counter()
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": PROMPT.format(q=question)}],
                max_tokens=800,
                temperature=0,
            )
            return resp.choices[0].message.content, time.perf_counter() - start, "ok"
        except Exception as e:
            err = str(e)[:150]
            if attempt < retries:
                time.sleep(3 * attempt)   # back off (e.g. rate limit) and retry
    return None, 0.0, f"error: {err}"


# ============================================================
# Replacement for Cell 5: resumes where it stopped and saves after every question.
# Run it as many times as you need; finished questions are never re-asked.
# ============================================================
import os

CKPT = "llama_gsm8k_results_partial.csv"

# `rows` still holds the finished questions if the Colab session wasn't restarted.
# If it was restarted (or `rows` is missing), reload from the checkpoint file instead.
if "rows" not in globals() or not rows:
    rows = pd.read_csv(CKPT).to_dict("records") if os.path.exists(CKPT) else []

done = {int(r["#"]) for r in rows}
print(f"Already done: {len(done)}/{N_QUESTIONS}")

try:
    for i, item in enumerate(ds, start=1):
        if i in done:
            continue
        truth = ground_truth(item["answer"])
        reply, secs, status = ask(item["question"])
        pred = extract_answer(reply)
        correct = "Yes" if (status == "ok" and is_correct(pred, truth)) else "No"
        rows.append({
            "#": i,
            "Question": item["question"],
            "Ground truth": truth,
            "Llama answer": pred,
            "Response time (s)": round(secs, 2),
            "Correct": correct,
            "Status": status,
            "Full response": reply,
        })
        pd.DataFrame(rows).to_csv(CKPT, index=False, encoding="utf-8-sig")   # checkpoint
        print(f"{i:>2}/{N_QUESTIONS}  truth={truth:<8} pred={str(pred):<8} {correct}  {secs:.2f}s  {status}")
except KeyboardInterrupt:
    print("\nStopped by you - progress is saved. Run this cell again to resume.")

df = pd.DataFrame(rows).sort_values("#").reset_index(drop=True)
df.to_csv("llama_gsm8k_results.csv", index=False, encoding="utf-8-sig")
print(f"\n{len(df)}/{N_QUESTIONS} done. Run Cell 6 once all {N_QUESTIONS} are finished.")

Already done: 3/50
 4/50  truth=5        pred=5        Yes  11.22s  ok
 5/50  truth=25       pred=25       Yes  3.32s  ok
 6/50  truth=452      pred=472      No  7.42s  ok
 7/50  truth=43       pred=43       Yes  14.96s  ok
 8/50  truth=34       pred=34       Yes  18.54s  ok
 9/50  truth=120      pred=120      Yes  2.37s  ok
10/50  truth=11       pred=22       No  6.88s  ok
11/50  truth=34       pred=27       No  10.47s  ok
12/50  truth=12       pred=12       Yes  3.20s  ok
13/50  truth=15       pred=24       No  48.16s  ok
14/50  truth=24       pred=24       Yes  21.04s  ok
15/50  truth=105      pred=105      Yes  27.12s  ok
16/50  truth=120      pred=120      Yes  4.59s  ok
17/50  truth=17       pred=17       Yes  8.78s  ok
18/50  truth=36       pred=36       Yes  4.15s  ok
19/50  truth=12       pred=12       Yes  7.19s  ok
20/50  truth=45       pred=45       Yes  3.74s  ok
21/50  truth=45       pred=45       Yes  12.70s  ok
22/50  truth=8        pred=8        Yes  6.68s  ok
23/50  t